# Fine-Tuning CodeT5 for React Native Code Generation

This notebook walks through **every step** of the RN-CodeGen project:

1. What is a Transformer and why CodeT5?
2. What is fine-tuning (vs. training from scratch)?
3. Dataset inspection
4. Tokenization — how text becomes numbers
5. The fine-tuning training loop (under the hood)
6. Running inference on the trained model
7. Evaluating with BLEU-4
8. Resume talking points

---
**Hardware:** runs on a free Colab T4 GPU (15 GB VRAM) in ~20 min.

## 0 — Install dependencies

In [ ]:
# Run once; restart the runtime after installation
!pip install -q transformers==4.40.0 datasets torch sacrebleu tqdm fastapi uvicorn pydantic

## 1 — Background: Transformers & CodeT5

### What is a Transformer?

A Transformer is a neural network architecture built entirely on **self-attention**.
Instead of processing tokens one-by-one (like an RNN), it processes the entire
sequence in parallel and learns which tokens should 'attend to' which others.

```
Input tokens  →  Embedding  →  [Attention + FFN] × N layers  →  Output
```

There are three families:

| Family | Examples | Good at |
|--------|----------|---------|
| Encoder-only | BERT, RoBERTa | Classification, understanding |
| Decoder-only | GPT-2, Llama | Text/code generation |
| **Encoder-decoder** | **T5, CodeT5, BART** | **Seq2seq: translation, summarisation, code gen** |

### Why CodeT5?

- **CodeT5** (Salesforce, 2021) is a T5 model pre-trained on **CodeSearchNet** — a
  dataset of 6 programming languages scraped from GitHub.
- The `codet5-small` variant has **60 M parameters** — small enough to fine-tune
  on a free GPU in under 30 minutes.
- Unlike GPT-2, CodeT5 is **encoder-decoder**: it takes a natural-language prompt
  as input and generates code as output — exactly what we need.

### What is fine-tuning?

Pre-training a model from scratch on billions of tokens costs millions of dollars.
**Fine-tuning** starts from a pre-trained checkpoint and continues training on a
small, domain-specific dataset (ours: React Native patterns).

```
CodeT5 (pre-trained on GitHub code)
         ↓  fine-tune on RN dataset (~20 examples, 10 epochs)
RN-CodeGen (generates React Native components)
```

The model already knows JavaScript syntax, React patterns, and component
structure from pre-training. Fine-tuning teaches it the *specific vocabulary*
of React Native: `StyleSheet`, `FlatList`, `TouchableOpacity`, `useNavigation`, etc.

## 2 — Dataset inspection

In [ ]:
import json
from pathlib import Path

# Load raw dataset
raw_path = Path('../data/rn_dataset.jsonl')
examples = [json.loads(line) for line in raw_path.read_text().splitlines() if line.strip()]

print(f'Total examples: {len(examples)}\n')
print('--- Example 1 ---')
print('PROMPT:', examples[0]['prompt'])
print('CODE:')
print(examples[0]['code'][:300], '...')

In [ ]:
# Analyse code lengths
import matplotlib.pyplot as plt

code_lengths = [len(ex['code'].split()) for ex in examples]
print(f'Min tokens : {min(code_lengths)}')
print(f'Max tokens : {max(code_lengths)}')
print(f'Avg tokens : {sum(code_lengths)/len(code_lengths):.0f}')

plt.figure(figsize=(8, 3))
plt.hist(code_lengths, bins=10, color='#6200ee', edgecolor='white')
plt.title('Distribution of code lengths (word tokens)')
plt.xlabel('Token count')
plt.ylabel('# examples')
plt.tight_layout()
plt.show()

## 3 — Preprocess: split train / val / test

In [ ]:
import subprocess
result = subprocess.run(['python', '../data/preprocess.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 4 — Tokenization deep-dive

Transformers do not operate on raw text.  Every character is first converted to
an integer *token id* via a **tokenizer**.  CodeT5 uses a **SentencePiece**
vocabulary of 32,000 sub-word tokens trained on code.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Salesforce/codet5-small')

prompt = 'Generate React Native code: Create a FlatList with pull-to-refresh'
encoded = tokenizer(prompt, return_tensors='pt')

print('Input text     :', prompt)
print('Token ids      :', encoded['input_ids'][0].tolist())
print('Decoded tokens :', [tokenizer.decode([t]) for t in encoded['input_ids'][0]])
print('Sequence length:', encoded['input_ids'].shape[1])

In [ ]:
# See how code is tokenized — sub-word units preserve identifier structure
snippet = 'const useFetch = (url) => { const [data, setData] = useState(null); }'
tokens = tokenizer.tokenize(snippet)
print('Sub-word tokens:', tokens)

## 5 — The PyTorch Dataset

In [ ]:
import sys
sys.path.insert(0, '..')

from training.dataset import RNCodeDataset

train_ds = RNCodeDataset('../data/processed/train.jsonl', tokenizer)
print(f'Train size: {len(train_ds)}')

sample = train_ds[0]
print('\nTensor shapes:')
for k, v in sample.items():
    print(f'  {k:20s} → {v.shape}  dtype={v.dtype}')

# Show labels — -100 positions are padding (ignored by loss)
label_ids = sample['labels']
non_pad = label_ids[label_ids != -100]
print(f'\nNon-padding label tokens: {len(non_pad)}')
print('First 10 label ids:', non_pad[:10].tolist())
print('Decoded           :', tokenizer.decode(non_pad[:10]))

## 6 — Training (what happens under the hood)

The training loop is standard seq2seq:

```
for each batch:
    1. Encoder reads the prompt → hidden states
    2. Decoder generates the code token-by-token,
       attending to encoder hidden states (cross-attention)
    3. Cross-entropy loss between predicted and reference tokens
    4. Backprop + AdamW weight update
```

Key techniques used:
- **Teacher forcing** — during training the decoder always receives the
  *ground-truth* previous token as input (not its own prediction), which
  stabilises training.
- **Mixed precision (fp16)** — half-precision activations halve VRAM usage.
- **Gradient accumulation** — simulates a larger batch by accumulating
  gradients over 4 mini-batches before stepping the optimiser.
- **Cosine LR schedule with warm-up** — learning rate rises linearly for the
  first 10 % of steps then decays following a cosine curve.

In [ ]:
# Visualise the LR schedule before training
import math
import matplotlib.pyplot as plt

total_steps  = 200
warmup_steps = 20
base_lr      = 5e-5

lrs = []
for step in range(total_steps):
    if step < warmup_steps:
        lr = base_lr * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))
    lrs.append(lr)

plt.figure(figsize=(8, 3))
plt.plot(lrs, color='#6200ee')
plt.axvline(warmup_steps, color='#e53935', linestyle='--', label='End of warmup')
plt.title('Cosine LR schedule with linear warm-up')
plt.xlabel('Optimiser step')
plt.ylabel('Learning rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── START TRAINING ────────────────────────────────────────────────────────────
# This cell runs the full fine-tuning script.
# Expected time on a T4 GPU: ~15-20 minutes for 10 epochs.
import subprocess
result = subprocess.run(['python', '../training/train.py'], text=True)
print('Return code:', result.returncode)

## 7 — Inference: generating React Native code

In [ ]:
from inference.generate import RNCodeGenerator

gen = RNCodeGenerator('../checkpoints')

prompts = [
    'Create a React Native FlatList with pull-to-refresh',
    'Create a custom hook for debounced search input',
    'Create a React Native animated button with press scale effect',
]

for prompt in prompts:
    print(f'\n>>> {prompt}')
    print('-' * 60)
    print(gen.generate(prompt))
    print()

## 8 — Evaluation: BLEU-4

**BLEU** (Bilingual Evaluation Understudy) measures n-gram overlap between
the generated text and a reference. For code generation:

- BLEU-4 score of **> 20** is considered reasonable for code generation tasks
- CodeBLEU (an extension) also accounts for syntactic / dataflow similarity
- Real-world CodeT5 fine-tuning on domain-specific data typically reaches
  BLEU-4 of **25–40** on held-out test sets

```
BLEU = BP × exp( sum( wn × log precision_n ) )

Where:
  precision_n = fraction of n-grams in hypothesis that appear in reference
  BP          = brevity penalty (penalises very short outputs)
  wn          = 1/4 for n=1..4 (uniform weights)
```

In [ ]:
# Demonstrate BLEU calculation manually before running full evaluation
import sacrebleu

hypothesis = 'const useFetch = url => { const [data, setData] = useState(null); return data; }'
reference  = 'const useFetch = (url) => { const [data, setData] = useState(null); return { data }; }'

score = sacrebleu.sentence_bleu(hypothesis, [reference])
print(f'Sentence BLEU-4: {score.score:.2f}')
print('\nNote: sentence-level BLEU is noisy; corpus-level BLEU (across all test examples) is the real metric.')

In [ ]:
# Run full evaluation on the test set
import subprocess
result = subprocess.run(
    ['python', '../evaluation/evaluate.py', '--model_dir', '../checkpoints'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Load and display per-example results
import json
with open('../evaluation/results.json') as f:
    results = json.load(f)

print('Metrics:', json.dumps(results['metrics'], indent=2))
print('\n--- Best example (highest token F1) ---')
best = max(results['per_example'], key=lambda x: x['token_f1'])
print('Prompt   :', best['prompt'])
print('Token F1 :', best['token_f1'])
print('\nPredicted:\n', best['predicted'][:400])

## 9 — Resume talking points

Here is how to describe this project concisely on a resume and in interviews:

---

### One-liner (resume bullet)
> *Fine-tuned Salesforce CodeT5 (60M-param encoder-decoder Transformer) on a
> curated React Native dataset; served via FastAPI; achieved BLEU-4 of ~XX on
> held-out test set.*

---

### Interview questions you can now answer confidently

**Q: What is fine-tuning and why did you choose it over training from scratch?**
> Fine-tuning starts from a checkpoint pre-trained on billions of code tokens
> (GitHub). The model already understands JS syntax and React patterns. Fine-tuning
> on my small RN dataset (~20 examples, expanded via preprocessing) adapts the
> model to RN-specific APIs in minutes rather than days, and with far less data.

**Q: Why CodeT5 and not GPT-2?**
> CodeT5 is encoder-decoder: the encoder reads the natural-language prompt and the
> decoder generates code conditioned on those representations. GPT-2 is
> decoder-only — it treats prompt + code as one concatenated sequence, which
> works but is less architecturally clean for translation-style tasks.

**Q: How did you prevent overfitting on such a small dataset?**
> Three mechanisms: (1) early stopping (patience = 3 eval rounds), (2) weight
> decay (L2 regularisation on all non-bias parameters), (3) cosine LR schedule
> with warm-up to avoid aggressive updates early in training.

**Q: What does BLEU-4 measure and what score did you get?**
> BLEU-4 measures the fraction of 4-grams in the generated code that appear in the
> reference, with a brevity penalty. Scores above 20 are considered reasonable for
> code generation. CodeT5 fine-tuned on domain-specific data typically reaches
> 25–40 on held-out test sets.

**Q: How did you serve the model?**
> FastAPI with an async lifespan handler that loads the model once at startup
> and keeps it in GPU memory. Each POST /generate request tokenises the prompt,
> runs beam search (width 4), and returns decoded code + latency in ms.